# ARX Order Search V1

Search này duyệt các order trọng điểm `ARX(na, nb, nk)` với:

- `na = 1..5`
- `nb = 1..5`
- `nk = 1..5`
- 16 augmented features
- z-score normalization theo train
- intercept
- clip free-run theo Q1%-Q99% train
- OLS cho search V1

Mục tiêu là tìm order tốt theo validation `FIT_sim`, sau đó xem test để kiểm tra generalization.


## 1. Import và cấu hình


In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd

WORK_DIR = Path.cwd()
PROJECT_ROOT = WORK_DIR.parent if WORK_DIR.name.startswith("ARX_Model_Version") else WORK_DIR
OUT_DIR = PROJECT_ROOT / "ARX_Model_VersionSearch"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from arx_pipeline import (
    DataConfig,
    SplitConfig,
    ModelConfig,
    load_or_generate_data,
    split_time_series,
    build_regression_matrix,
    estimate_ols,
    compute_metrics,
    simulate_arx,
    simulate_arx_n_step,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

NA_LIST = [1, 2, 3, 4, 5]
NB_LIST = [1, 2, 3, 4, 5]
NK_LIST = [1, 2, 3, 4, 5]
CLIP_QUANTILES = (0.01, 0.99)
N_STEP = 12


## 2. Load data và tạo augmented features


In [2]:
DATA_CONFIG = DataConfig(
    csv_path=PROJECT_ROOT / "greenhouse_data.csv",
    generator_script_path=PROJECT_ROOT / "data_generator.py",
    force_regenerate_from_script=False,
    auto_save_generated_csv=True,
)

SPLIT_CONFIG = SplitConfig(train_ratio=0.60, val_ratio=0.20)

BASELINE_INPUT_COLS = (
    "Temperature",
    "Humidity",
    "Light",
    "Drip",
    "Mist",
    "Fan",
)

AUGMENTED_INPUT_COLS = (
    *BASELINE_INPUT_COLS,
    "Light_log",
    "Temp_x_Humi",
    "Temp_x_Light",
    "Humi_x_Light",
    "SP_Center",
    "SP_Width",
    "Month_sin",
    "Month_cos",
    "Season_sin",
    "Season_cos",
)


def build_augmented_df(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    df["Light_log"] = np.log1p(df["Light"].clip(lower=0))
    df["Temp_x_Humi"] = df["Temperature"] * df["Humidity"]
    df["Temp_x_Light"] = df["Temperature"] * df["Light_log"]
    df["Humi_x_Light"] = df["Humidity"] * df["Light_log"]
    df["SP_Center"] = 0.5 * (df["Soil_Low_SP"] + df["Soil_High_SP"])
    df["SP_Width"] = df["Soil_High_SP"] - df["Soil_Low_SP"]
    df["Month_sin"] = np.sin(2.0 * np.pi * df["Month"] / 12.0)
    df["Month_cos"] = np.cos(2.0 * np.pi * df["Month"] / 12.0)
    season_map = {"spring": 0, "summer": 1, "autumn": 2, "winter": 3}
    season_num = df["Season"].map(season_map).fillna(0).astype(float)
    df["Season_sin"] = np.sin(2.0 * np.pi * season_num / 4.0)
    df["Season_cos"] = np.cos(2.0 * np.pi * season_num / 4.0)
    return df


df_full, true_params, data_source = load_or_generate_data(DATA_CONFIG)
df_aug = build_augmented_df(df_full)
df_train, df_val, df_test = split_time_series(df_aug, SPLIT_CONFIG)

print("Data source:", data_source)
print("Rows:", len(df_full), len(df_train), len(df_val), len(df_test))
display(df_train[list(AUGMENTED_INPUT_COLS)].head())


Data source: CSV:greenhouse_data.csv
Rows: 105120 63072 21024 21024


,Temperature,Humidity,Light,Drip,Mist,Fan,Light_log,Temp_x_Humi,Temp_x_Light,Humi_x_Light,SP_Center,SP_Width,Month_sin,Month_cos,Season_sin,Season_cos
0,20.778462,83.333673,1.168667,0.0,0.0,0.0,0.774113,1731.545561,16.084869,64.509646,55.5,9.0,0.5,0.866025,-1.0,-1.836970e-16
1,19.725025,82.264886,2.256788,0.0,0.0,0.0,1.180741,1622.676903,23.290155,97.133564,55.5,9.0,0.5,0.866025,-1.0,-1.836970e-16
2,19.696345,82.314817,20.575650,0.0,0.0,0.0,3.071565,1621.301048,60.498612,252.835341,55.5,9.0,0.5,0.866025,-1.0,-1.836970e-16
3,19.701227,82.287578,25.108612,0.0,0.0,0.0,3.262265,1621.166241,64.270627,268.443904,55.5,9.0,0.5,0.866025,-1.0,-1.836970e-16
4,19.837379,83.482156,7.002633,0.0,0.0,0.0,2.079771,1656.067143,41.257198,173.623737,55.5,9.0,0.5,0.866025,-1.0,-1.836970e-16


## 3. Z-score và clip theo train


In [3]:
SCALE_COLS = ("Soil_Moisture", *AUGMENTED_INPUT_COLS)
scale_stats = {}
for col in SCALE_COLS:
    mean = float(df_train[col].astype(float).mean())
    std = float(df_train[col].astype(float).std(ddof=0))
    if not np.isfinite(std) or std < 1e-12:
        std = 1.0
    scale_stats[col] = {"mean": mean, "std": std}


def apply_zscore(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    for col, st in scale_stats.items():
        df_out[col] = (df_out[col].astype(float) - st["mean"]) / st["std"]
    return df_out


def inverse_y(values: np.ndarray) -> np.ndarray:
    st = scale_stats["Soil_Moisture"]
    return np.asarray(values, dtype=float) * st["std"] + st["mean"]


clip_low_real = float(df_train["Soil_Moisture"].quantile(CLIP_QUANTILES[0]))
clip_high_real = float(df_train["Soil_Moisture"].quantile(CLIP_QUANTILES[1]))
y_scale = scale_stats["Soil_Moisture"]
clip_bounds_scaled = (
    float((clip_low_real - y_scale["mean"]) / y_scale["std"]),
    float((clip_high_real - y_scale["mean"]) / y_scale["std"]),
)

df_train_z = apply_zscore(df_train)
df_val_z = apply_zscore(df_val)
df_test_z = apply_zscore(df_test)

pd.Series({
    "clip_low_real": clip_low_real,
    "clip_high_real": clip_high_real,
    "clip_low_scaled": clip_bounds_scaled[0],
    "clip_high_scaled": clip_bounds_scaled[1],
})


clip_low_real       50.546579
clip_high_real      64.612728
clip_low_scaled     -2.064203
clip_high_scaled     2.130330
dtype: float64

## 4. Lightweight order evaluation


In [4]:
def metrics_for_split(df_split_z: pd.DataFrame, theta: np.ndarray, cfg: ModelConfig) -> dict[str, float]:
    x_mat, y_vec = build_regression_matrix(df_split_z, cfg)
    y_pred_1 = x_mat @ theta
    y_pred_sim, y_true_sim = simulate_arx(df_split_z, theta, cfg)
    y_pred_12, _ = simulate_arx_n_step(df_split_z, theta, n_steps=N_STEP, model_config=cfg)

    m1 = compute_metrics(inverse_y(y_vec), inverse_y(y_pred_1), len(theta))
    m12 = compute_metrics(inverse_y(y_true_sim), inverse_y(y_pred_12), len(theta))
    msim = compute_metrics(inverse_y(y_true_sim), inverse_y(y_pred_sim), len(theta))
    return {
        "FIT_1step": m1["FIT"],
        "RMSE_1step": m1["RMSE"],
        "FIT_12step": m12["FIT"],
        "RMSE_12step": m12["RMSE"],
        "FIT_sim": msim["FIT"],
        "RMSE_sim": msim["RMSE"],
        "Bias_sim": msim["Bias"],
    }


def evaluate_order(na: int, nb: int, nk: int) -> dict:
    cfg = ModelConfig(
        na=na,
        nb=nb,
        nk=nk,
        include_intercept=True,
        input_cols=AUGMENTED_INPUT_COLS,
        output_col="Soil_Moisture",
        simulation_clip=clip_bounds_scaled,
    )
    x_train, y_train = build_regression_matrix(df_train_z, cfg)
    theta, _, sigma2 = estimate_ols(x_train, y_train)
    val = metrics_for_split(df_val_z, theta, cfg)
    test = metrics_for_split(df_test_z, theta, cfg)
    return {
        "na": na,
        "nb": nb,
        "nk": nk,
        "order": f"({na},{nb},{nk})",
        "n_params": len(theta),
        "rank_x_train": int(np.linalg.matrix_rank(x_train)),
        "sigma2_train": float(sigma2),
        "val_FIT_1step": val["FIT_1step"],
        "val_FIT_12step": val["FIT_12step"],
        "val_FIT_sim": val["FIT_sim"],
        "val_RMSE_sim": val["RMSE_sim"],
        "test_FIT_1step": test["FIT_1step"],
        "test_FIT_12step": test["FIT_12step"],
        "test_FIT_sim": test["FIT_sim"],
        "test_RMSE_sim": test["RMSE_sim"],
        "test_Bias_sim": test["Bias_sim"],
    }


## 5. Chạy grid search


In [5]:
start = time.time()
rows = []
errors = []
total = len(NA_LIST) * len(NB_LIST) * len(NK_LIST)
done = 0

for na in NA_LIST:
    for nb in NB_LIST:
        for nk in NK_LIST:
            done += 1
            try:
                rows.append(evaluate_order(na, nb, nk))
            except Exception as exc:
                errors.append({"na": na, "nb": nb, "nk": nk, "error": str(exc)})
            if done % 10 == 0 or done == total:
                print(f"{done}/{total} done")

search_df = pd.DataFrame(rows).sort_values(
    ["val_FIT_sim", "val_FIT_12step", "val_FIT_1step", "n_params"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

elapsed_seconds = time.time() - start
print("Elapsed seconds:", round(elapsed_seconds, 2))
print("Errors:", len(errors))
search_df.round(4)


10/125 done


20/125 done


30/125 done


40/125 done


50/125 done


...

100/125 done


110/125 done


120/125 done


125/125 done
Elapsed seconds: 938.87
Errors: 0


,na,nb,nk,order,n_params,rank_x_train,sigma2_train,val_FIT_1step,val_FIT_12step,val_FIT_sim,val_RMSE_sim,test_FIT_1step,test_FIT_12step,test_FIT_sim,test_RMSE_sim,test_Bias_sim
0,1,5,2,"(1,5,2)",82,77,0.0142,86.6057,70.3347,68.9559,0.9320,86.1514,67.8978,66.8337,0.9661,0.0162
1,5,1,2,"(5,1,2)",22,21,0.0149,86.2982,69.8051,68.8411,0.9355,85.8492,67.1601,66.4176,0.9782,0.0033
2,2,5,2,"(2,5,2)",83,78,0.0142,86.6186,69.9361,68.7952,0.9369,86.1597,67.2810,66.4020,0.9787,0.0169
3,5,3,2,"(5,3,2)",54,51,0.0147,86.3764,69.8777,68.7876,0.9371,85.9346,67.2998,66.4922,0.9760,0.0079
4,4,5,2,"(4,5,2)",85,80,0.0141,86.6512,69.7069,68.6769,0.9404,86.1803,66.9463,66.1571,0.9858,0.0170
5,5,5,2,"(5,5,2)",86,81,0.0141,86.6546,69.4785,68.6020,0.9427,86.1813,66.5545,65.8789,0.9939,0.0160
6,3,5,2,"(3,5,2)",84,79,0.0141,86.6437,69.3549,68.5169,0.9452,86.1729,66.3892,65.7475,0.9978,0.0161
7,5,2,2,"(5,2,2)",38,36,0.0148,86.3421,68.8096,68.3247,0.9510,85.8846,65.6503,65.2975,1.0108,0.0124
8,1,1,2,"(1,1,2)",18,17,0.0159,85.7876,69.0290,68.2521,0.9531,85.4743,66.7017,66.2635,0.9826,0.0249
9,1,3,2,"(1,3,2)",50,47,0.0159,85.7948,69.0043,68.2398,0.9535,85.4740,66.5693,66.1694,0.9854,0.0219


## 6. Bảng so sánh với các version hiện có


In [6]:
version_rows = []

version_sources = [
    ("221 V1", PROJECT_ROOT / "ARX_Model_Version 221" / "arx_baseline_v1.json"),
    ("221 V6", PROJECT_ROOT / "ARX_Model_Version 221" / "arx_baseline_v6.json"),
    ("512 V1", PROJECT_ROOT / "ARX_Model_Version 512" / "arx_512_v1.json"),
    ("512 V6", PROJECT_ROOT / "ARX_Model_Version 512" / "arx_512_v6.json"),
]

for label, path in version_sources:
    if not path.exists():
        continue
    with path.open("r", encoding="utf-8") as f:
        artifact = json.load(f)
    version_rows.append({
        "model": label,
        "order": f"({artifact['model_config']['na']},{artifact['model_config']['nb']},{artifact['model_config']['nk']})",
        "n_inputs": len(artifact["model_config"]["input_cols"]),
        "clip": artifact.get("simulation_clip", {}).get("enabled", artifact["model_config"].get("simulation_clip") is not None),
        "val_FIT_sim": artifact["metrics"]["validation"]["fit_sim"],
        "test_FIT_sim": artifact["metrics"]["test"]["fit_sim"],
    })

best = search_df.iloc[0]
version_rows.append({
    "model": "Search V1 best by val",
    "order": best["order"],
    "n_inputs": len(AUGMENTED_INPUT_COLS),
    "clip": True,
    "val_FIT_sim": float(best["val_FIT_sim"]),
    "test_FIT_sim": float(best["test_FIT_sim"]),
})

comparison_df = pd.DataFrame(version_rows)
comparison_df["test_gain_vs_221_v1"] = comparison_df["test_FIT_sim"] - comparison_df.loc[0, "test_FIT_sim"]
comparison_df.round(4)


,model,order,n_inputs,clip,val_FIT_sim,test_FIT_sim,test_gain_vs_221_v1
0,221 V1,"(2,2,1)",6,False,42.9588,43.8749,0.0000
1,221 V6,"(2,2,1)",16,True,48.7386,52.7359,8.8609
2,512 V1,"(5,1,2)",6,False,45.3080,34.1171,-9.7579
3,512 V6,"(5,1,2)",16,True,68.8411,66.4176,22.5426
4,Search V1 best by val,"(1,5,2)",16,True,68.9559,66.8337,22.9588


## 7. Lưu artifact


In [7]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


best = search_df.iloc[0].to_dict()
artifact = {
    "version": "order_search_v1",
    "search_space": {
        "na": NA_LIST,
        "nb": NB_LIST,
        "nk": NK_LIST,
        "n_candidates_requested": len(NA_LIST) * len(NB_LIST) * len(NK_LIST),
        "n_candidates_success": int(len(search_df)),
        "n_errors": int(len(errors)),
        "errors": errors,
    },
    "fixed_config": {
        "input_cols": list(AUGMENTED_INPUT_COLS),
        "include_intercept": True,
        "normalization": "zscore fitted on train",
        "simulation_clip_scaled": list(clip_bounds_scaled),
        "simulation_clip_real": [clip_low_real, clip_high_real],
        "estimator": "OLS",
        "selection_metric": "max validation FIT_sim",
    },
    "best_by_validation": best,
    "top_20": search_df.head(20).to_dict(orient="records"),
    "all_results": search_df.to_dict(orient="records"),
    "comparison_table": comparison_df.to_dict(orient="records"),
    "elapsed_seconds": elapsed_seconds,
}

OUT_DIR.mkdir(exist_ok=True)
out_path = OUT_DIR / "arx_order_search_v1.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(json_ready(artifact), f, indent=2)
    f.write("\n")

out_path


WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/ARX_Model_VersionSearch/arx_order_search_v1.json')